In [1]:
# Imports & Config

import subprocess
subprocess.run(['pip', 'install', 'ultralytics', '-q'])

import os, json, shutil, torch
import numpy as np
from ultralytics import YOLO
import warnings
warnings.filterwarnings('ignore')

DEVICE    = 'cuda' if torch.cuda.is_available() else 'cpu'
NUM_GPUS  = torch.cuda.device_count()

print(f"Device  : {DEVICE}")
print(f"GPUs    : {NUM_GPUS}")

DATA_PATH = '/kaggle/input/datasets/pankajdeopaiiitb/vr-yolo-dataset/data.yaml'
SAVE_DIR  = '/kaggle/working/yolo_runs/'
RUN_NAME = 'yolov8s_apparel_scratch'
RUN_DIR = SAVE_DIR + RUN_NAME + '/weights/'

os.makedirs(SAVE_DIR, exist_ok=True)

IDX_TO_NAME = {
    0: 'short_sleeve_top',
    1: 'trousers',
    2: 'shorts',
    3: 'long_sleeve_top',
    4: 'skirt'
}

print("Config ready")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 20.7 MB/s eta 0:00:00
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Device  : cuda
GPUs    : 2
Config ready


In [2]:
# Load Model

model = YOLO("yolov8s-seg.yaml")
print("Model loaded: YOLOv8s-seg")
print(f"Parameters : {sum(p.numel() for p in model.model.parameters())/1e6:.2f}M")

Model loaded: YOLOv8s-seg
Parameters : 11.82M


In [3]:
# Training

RUN_DIR     = SAVE_DIR + RUN_NAME + '/weights/'
resume_flag = os.path.exists(RUN_DIR + 'last.pt')

if resume_flag:
    print(f"Resuming from checkpoint: {RUN_DIR}last.pt")
else:
    print("Starting fresh training")

# Use both GPUs if available
device_arg = list(range(NUM_GPUS)) if NUM_GPUS > 1 else 0
print(f"Training on device: {device_arg}")

results = model.train(
    data        = DATA_PATH,
    epochs      = 25,         
    imgsz       = 640, 
    batch       = 32, 
    device      = device_arg,
    project     = SAVE_DIR,
    name        = RUN_NAME,
    exist_ok    = True,
    resume      = resume_flag,
    patience    = 15,           # early stopping
    save        = True,
    save_period = 5,            # save every 5 epochs
    workers     = 4,
    verbose     = True,
    # Augmentation
    hsv_h       = 0.015,
    hsv_s       = 0.7,
    hsv_v       = 0.4,
    fliplr      = 0.5,
    mosaic      = 1.0,
    amp         = True,
    pretrained  = False,
)

print("\nTraining complete!")
print(f"Best model: {SAVE_DIR}{RUN_NAME}/weights/best.pt")

Starting fresh training
Training on device: [0, 1]
Ultralytics 8.4.31 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
                                                       CUDA:1 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/input/datasets/pankajdeopaiiitb/vr-yolo-dataset/data.yaml, degrees=0.0, deterministic=True, device=0,1, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=25, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8s-

In [4]:
# Evaluate

best_model_path = SAVE_DIR + RUN_NAME + '/weights/best.pt'
best_model      = YOLO(best_model_path)

print("Running validation on best model...")
metrics = best_model.val(
    data    = DATA_PATH,
    imgsz   = 640,
    batch   = 32,
    device  = device_arg,
    verbose = True
)

# ── Detection Metrics ──────────────────────────────────────────
print("\n── Detection Metrics ──")
print(f"mAP@0.5        : {metrics.box.map50:.4f}")
print(f"mAP@0.5:0.95   : {metrics.box.map:.4f}")
print(f"Precision      : {metrics.box.mp:.4f}")
print(f"Recall         : {metrics.box.mr:.4f}")

# ── Segmentation Metrics ───────────────────────────────────────
print("\n── Segmentation Metrics ──")
print(f"Mask mAP@0.5      : {metrics.seg.map50:.4f}")
print(f"Mask mAP@0.5:0.95 : {metrics.seg.map:.4f}")
print(f"Mask Precision    : {metrics.seg.mp:.4f}")
print(f"Mask Recall       : {metrics.seg.mr:.4f}")

# ── Per-class Detection mAP ────────────────────────────────────
print("\n── Per-class Detection mAP@0.5 ──")
print(f"{'Class':<20} {'mAP@0.5':>10} {'mAP@0.5:0.95':>14}")
print("-" * 46)
for i, name in IDX_TO_NAME.items():
    ap50    = metrics.box.ap50[i]
    ap50_95 = metrics.box.ap[i]
    print(f"{name:<20} {ap50:>10.4f} {ap50_95:>14.4f}")

# ── Per-class Segmentation mAP ─────────────────────────────────
print("\n── Per-class Segmentation mAP@0.5 ──")
print(f"{'Class':<20} {'mAP@0.5':>10} {'mAP@0.5:0.95':>14}")
print("-" * 46)
for i, name in IDX_TO_NAME.items():
    seg_ap50    = metrics.seg.ap50[i]
    seg_ap50_95 = metrics.seg.ap[i]
    print(f"{name:<20} {seg_ap50:>10.4f} {seg_ap50_95:>14.4f}")

Running validation on best model...
Ultralytics 8.4.31 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
                                                       CUDA:1 (Tesla T4, 14913MiB)
YOLOv8s-seg summary (fused): 86 layers, 11,781,535 parameters, 0 gradients, 39.9 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 21.5±14.0 MB/s, size: 29.0 KB)
val: Scanning /kaggle/input/datasets/pankajdeopaiiitb/vr-yolo-dataset/labels/val... 6932 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 6932/6932 383.6it/s 18.1s
WARNING ⚠️ val: Cache directory /kaggle/input/datasets/pankajdeopaiiitb/vr-yolo-dataset/labels is not writable, cache not saved.
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95)     Mask(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 217/217 1.9it/s 1:52
                   all       6932       9950      0.836       0.83      0.898      0.764      0.825      0.821      0.884      0.686
      short_sleeve_top     

In [5]:
# Save Metrics & Upload

metrics_summary = {
    'detection': {
        'mAP50'    : float(metrics.box.map50),
        'mAP50_95' : float(metrics.box.map),
        'precision': float(metrics.box.mp),
        'recall'   : float(metrics.box.mr),
        'per_class': {
            IDX_TO_NAME[i]: {
                'ap50'   : float(metrics.box.ap50[i]),
                'ap50_95': float(metrics.box.ap[i])
            } for i in range(5)
        }
    },
    'segmentation': {
        'mAP50'    : float(metrics.seg.map50),
        'mAP50_95' : float(metrics.seg.map),
        'precision': float(metrics.seg.mp),
        'recall'   : float(metrics.seg.mr),
        'per_class': {
            IDX_TO_NAME[i]: {
                'ap50'   : float(metrics.seg.ap50[i]),
                'ap50_95': float(metrics.seg.ap[i])
            } for i in range(5)
        }
    }
}

with open('/kaggle/working/yolo_scratch_metrics.json', 'w') as f:
    json.dump(metrics_summary, f, indent=2)
print("Metrics saved")

# ── Upload to Kaggle Dataset ───────────────────────────────────
UPLOAD_DIR = '/kaggle/working/yolo_scratch_upload/' 
os.makedirs(UPLOAD_DIR, exist_ok=True)

shutil.copy(SAVE_DIR + RUN_NAME + '/weights/best.pt',
            UPLOAD_DIR + 'yolov8s_scratch_best.pt') 

shutil.copy('/kaggle/working/yolo_scratch_metrics.json',
            UPLOAD_DIR + 'metrics_scratch.json')

# Save label mapping
label_mapping = {
    "short_sleeve_top": 0,
    "trousers"        : 1,
    "shorts"          : 2,
    "long_sleeve_top" : 3,
    "skirt"           : 4,
    "note"            : "0-based indexing, no background class for YOLO, SCRATCH training"  # changed
}
with open(UPLOAD_DIR + 'label_mapping.json', 'w') as f:
    json.dump(label_mapping, f, indent=2)

metadata = {
    "title"   : "vr-yolo-scratch-model",          # changed
    "id"      : "pankajdeopaiiitb/vr-yolo-scratch-model",  # changed
    "licenses": [{"name": "CC0-1.0"}]
}
with open(UPLOAD_DIR + 'dataset-metadata.json', 'w') as f:
    json.dump(metadata, f)

result = subprocess.run(
    ['kaggle', 'datasets', 'create', '-p', UPLOAD_DIR, '--dir-mode', 'zip'],
    capture_output=True, text=True
)

print(result.stdout)
print(result.stderr)

Metrics saved
Starting upload for file metrics_scratch.json
Upload successful: metrics_scratch.json (1KB)
Starting upload for file label_mapping.json
Upload successful: label_mapping.json (175B)
Starting upload for file yolov8s_scratch_best.pt
Upload successful: yolov8s_scratch_best.pt (23MB)
Your private Dataset is being created. Please check progress at https://www.kaggle.com/datasets/pankajdeopaiiitb/vr-yolo-scratch-model


  0%|          | 0.00/1.39k [00:00<?, ?B/s]
100%|██████████| 1.39k/1.39k [00:00<00:00, 3.72kB/s]

  0%|          | 0.00/175 [00:00<?, ?B/s]
100%|██████████| 175/175 [00:00<00:00, 531B/s]

  0%|          | 0.00/22.7M [00:00<?, ?B/s]
 41%|████      | 9.31M/22.7M [00:00<00:00, 80.4MB/s]
 76%|███████▌  | 17.3M/22.7M [00:00<00:00, 79.6MB/s]
100%|██████████| 22.7M/22.7M [00:00<00:00, 37.9MB/s]

